# MetroPT-3: external failure anchors and censored observations

Evidence class: one compressor sensor stream with externally reported air-leak intervals. The tracked interval manifest is an external-anchor declaration; observations outside those intervals remain unlabeled/censored, not negative. The notebook invokes the existing verifier and charts only its verified anchor-coverage and cadence projections.

Expected local input: `data/MetroPT-3(AirCompressor).csv`. The default anchor manifest is `examples/metropt3/failure_intervals.json`. Override the data root with `ISOPRAX_DATA_DIR` before starting Jupyter.

In [ ]:
import os
import sys
from pathlib import Path

working_directory = Path.cwd().resolve()
working_root = next(
    (
        candidate
        for candidate in (working_directory, *working_directory.parents)
        if (candidate / "pyproject.toml").is_file() and (candidate / "isoprax").is_dir()
    ),
    None,
)
configured_root = os.environ.get("ISOPRAX_REPO_ROOT")
REPO_ROOT = (
    Path(configured_root).expanduser().resolve() if configured_root else working_root
)
if (
    REPO_ROOT is None
    or not (REPO_ROOT / "pyproject.toml").is_file()
    or not (REPO_ROOT / "isoprax").is_dir()
):
    raise RuntimeError(
        "Launch from this checkout or set ISOPRAX_REPO_ROOT to its repository root."
    )
if configured_root and working_root is not None and REPO_ROOT != working_root:
    raise RuntimeError(
        "ISOPRAX_REPO_ROOT does not match the notebook's checkout directory."
    )
repo_path = str(REPO_ROOT)
if repo_path in sys.path:
    sys.path.remove(repo_path)
sys.path.insert(0, repo_path)

import isoprax

PACKAGE_ROOT = Path(isoprax.__file__).resolve().parents[1]
if PACKAGE_ROOT != REPO_ROOT:
    raise RuntimeError(
        "The selected Python kernel does not import Isoprax from this checkout."
    )

from notebooks._support import (
    data_root_from_environment,
    display_review,
    review_dataset,
)

DATA_ROOT = data_root_from_environment(REPO_ROOT)
CSV_PATH = DATA_ROOT / "MetroPT-3(AirCompressor).csv"
INTERVALS_PATH = REPO_ROOT / "examples" / "metropt3" / "failure_intervals.json"
print(f"Python: {sys.version.split()[0]} ({sys.executable})")
print(f"Local sensor CSV: {CSV_PATH}")
print(f"Anchor manifest: {INTERVALS_PATH}")

In [ ]:
review = review_dataset(
    "metropt3",
    {
        "csv": CSV_PATH,
        "intervals": INTERVALS_PATH,
    },
    REPO_ROOT,
)
display_review(review)

The cadence table summarizes actual adjacent timestamp gaps from the verified file. Sensor summaries are streamed in bounded chunks: hourly means across the stream and minute means within six-hour context windows around the external anchors. Shading marks only each reported interval; context margins are not labels. Uncovered observations remain censored, not clean/negative, so this notebook deliberately computes no supervised predictor score.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from isoprax.metropt3 import read_metropt3_intervals
from notebooks import _analysis

if review.verification.status in {"verified", "verified_with_warnings"}:
    report = review.verification.report
    anchors = read_metropt3_intervals(INTERVALS_PATH)
    views = _analysis.read_metropt3_views(
        CSV_PATH, anchors, chunksize=100_000, window_margin_seconds=21_600
    )
    display(Markdown("## Observed stream and timestamp cadence"))
    gap_counts = {
        float(gap): int(count) for gap, count in report["gap_counts_seconds"].items()
    }
    gap_total = sum(gap_counts.values())
    if gap_total:
        ordered_gaps = sorted(gap_counts.items())
        median_rank = (gap_total - 1) // 2
        cumulative = 0
        for gap, count in ordered_gaps:
            cumulative += count
            if cumulative > median_rank:
                median_gap = gap
                break
        modal_gap, modal_count = max(
            gap_counts.items(), key=lambda item: (item[1], -item[0])
        )
        print(
            f"Observed adjacent timestamp gaps: {gap_total:,}; weighted median: {median_gap:g} s; modal gap: {modal_gap:g} s ({modal_count:,} occurrences)"
        )
        display(
            pd.DataFrame(
                [
                    {"gap_seconds": gap, "count": count, "share": count / gap_total}
                    for gap, count in gap_counts.items()
                ]
            )
            .sort_values("count", ascending=False)
            .head(12)
        )
    display(pd.DataFrame([report["observed"]]))

    anchor_rows = [
        {
            "anchor_id": anchor.anchor_id,
            "reported_start": anchor.start,
            "reported_end": anchor.end,
            "covered_rows": dict(report["interval_coverage"]).get(anchor.anchor_id, 0),
            "source_reference": anchor.source_reference,
        }
        for anchor in anchors
    ]
    display(pd.DataFrame(anchor_rows).set_index("anchor_id"))

    display(Markdown("## Chunk-reduced stream overview (hourly means)"))
    figure, axis = plt.subplots(figsize=(14, 5), constrained_layout=True)
    for sensor in ("TP2", "Motor_current"):
        axis.plot(views.hourly.index, views.hourly[sensor], label=sensor, linewidth=0.8)
    for anchor in anchors:
        axis.axvspan(anchor.start, anchor.end, color="#b84a4a", alpha=0.16)
    axis.set_title("Hourly sensor means with externally reported intervals shaded")
    axis.set_ylabel("Sensor value (native units)")
    axis.legend()
    display(figure)
    plt.close(figure)

    display(
        Markdown("## Anchor context (minute means; ±6 hours are visual context only)")
    )
    figure, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    for anchor, axis in zip(anchors, axes.flat):
        minute_view = views.interval_minutes[anchor.anchor_id]
        for sensor in ("TP2", "Motor_current"):
            if sensor in minute_view:
                axis.plot(
                    minute_view.index, minute_view[sensor], label=sensor, linewidth=0.9
                )
        axis.axvspan(
            anchor.start,
            anchor.end,
            color="#b84a4a",
            alpha=0.2,
            label="reported interval",
        )
        axis.set_title(f"{anchor.anchor_id}: {anchor.start} to {anchor.end}")
        axis.legend(fontsize=8)
    display(figure)
    plt.close(figure)
    display(
        Markdown(
            "No row-level negatives or predictive metrics are computed. Outside the shaded external intervals, event status remains censored/unknown."
        )
    )
else:
    display(
        Markdown(
            "Exploration is gated: raw rows are streamed only after the canonical verifier succeeds."
        )
    )